<a href="https://colab.research.google.com/github/MEHWISH310/Animal-Detection-Using-YOLOe-26s-seg/blob/main/yolo_sound.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install ultralytics opencv-python -q
!apt-get install -y ffmpeg libsndfile1 -q

import cv2
import numpy as np
import wave
import threading
import subprocess
import time
from ultralytics import YOLO
print("Libraries loaded ✅")

In [ ]:
import numpy as np
import wave

def generate_beep(filename, frequency=880, duration=0.4, volume=0.9, sample_rate=44100):
    n_samples = int(sample_rate * duration)
    t         = np.linspace(0, duration, n_samples)
    wave_data = np.sin(2 * np.pi * frequency * t)
    fade      = np.linspace(1.0, 0.0, n_samples)
    wave_data = (wave_data * fade * volume * 32767).astype(np.int16)
    with wave.open(filename, 'w') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(wave_data.tobytes())

generate_beep("alert_animal.wav", frequency=880, duration=0.4)
generate_beep("alert_clear.wav",  frequency=660, duration=0.3)
print("Alert sounds generated ✅")

In [ ]:
VIDEO_INPUT  = "night.mp4"
VIDEO_OUTPUT = "animal_detection_output.avi"
ANIMAL_PROMPTS = [
    "cow", "sheep", "duck", "goat", "dog", "cat", "horse", "deer",
    "elephant", "fox", "rabbit", "bird", "camel", "buffalo", "zebra", "tiger"
]

CONF_THRESHOLD     = 0.40
REGION_LINE_Y      = None
TEXT_COLOR         = (255, 255, 255)
ALERT_COLOR        = (0, 0, 180)

animal_present     = False
CLEAR_GRACE_FRAMES = 75
no_animal_counter  = 0

PALETTE = [
    (0,0,255),(0,165,255),(0,255,255),(255,0,0),
    (255,0,255),(0,255,0),(128,0,128),(0,128,255),
]
def get_color(cls_id):
    return PALETTE[cls_id % len(PALETTE)]

print(f"Config set ✅  —  {len(ANIMAL_PROMPTS)} prompts")
print("Prompts:", ANIMAL_PROMPTS)

In [ ]:
model = YOLO("yoloe-26s-seg.pt")

text_embeddings = model.get_text_pe(ANIMAL_PROMPTS)
model.set_classes(ANIMAL_PROMPTS, text_embeddings)
CLASS_NAMES = model.names
print("Model loaded ✅")
print("True class mapping:")
for cid, cname in CLASS_NAMES.items():
    expected = ANIMAL_PROMPTS[cid] if cid < len(ANIMAL_PROMPTS) else "?"
    flag     = "✅" if cname == expected else "❌ MISMATCH"
    print(f"  [{cid:02d}] model='{cname}'  list='{expected}'  {flag}")

In [ ]:
cap = cv2.VideoCapture(VIDEO_INPUT)
assert cap.isOpened(), f'Cannot open "{VIDEO_INPUT}"'

w            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps          = cap.get(cv2.CAP_PROP_FPS) or 25
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video → {w}x{h} @ {fps:.1f} FPS | {total_frames} frames")

video_writer = cv2.VideoWriter(
    VIDEO_OUTPUT,
    cv2.VideoWriter_fourcc(*"XVID"),
    fps, (w, h)
)
print("VideoWriter ready ✅")

In [ ]:
total_animals_seen   = {}
per_class_count      = {}
track_class_history  = {}
frame_count          = 0
alert_timestamps_sec = []

def get_true_label(cls_id, results):
    if hasattr(results[0], 'names') and cls_id in results[0].names:
        return results[0].names[cls_id]
    if cls_id in CLASS_NAMES:
        return CLASS_NAMES[cls_id]
    if cls_id < len(ANIMAL_PROMPTS):
        return ANIMAL_PROMPTS[cls_id]
    return "unknown"

def play_sound_async(filepath, delay=0.0):
    def _play():
        try:
            if delay:
                time.sleep(delay)
            subprocess.Popen(
                ["ffplay", "-nodisp", "-autoexit", "-loglevel", "quiet", filepath],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
        except Exception as e:
            print(f"[Audio] {e}")
    threading.Thread(target=_play, daemon=True).start()

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    frame_count += 1

    results = model.track(
        frame,
        conf    = CONF_THRESHOLD,
        iou     = 0.45,
        persist = True,
        verbose = False,
    )

    detections_this_frame = []

    if results[0].boxes is not None:
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_id   = int(box.cls[0])
            conf     = float(box.conf[0])
            track_id = int(box.id[0]) if box.id is not None else -1

            label = get_true_label(cls_id, results)
            if track_id != -1:
                if track_id not in track_class_history:
                    track_class_history[track_id] = {}
                track_class_history[track_id][label] = \
                    track_class_history[track_id].get(label, 0) + 1
                label = max(track_class_history[track_id],
                            key=track_class_history[track_id].get)

            if REGION_LINE_Y and y2 < REGION_LINE_Y:
                continue

            detections_this_frame.append(
                (x1, y1, x2, y2, label, conf, track_id, cls_id)
            )

            if track_id != -1:
                key = (cls_id, track_id)
                total_animals_seen[key] = label
                per_class_count.setdefault(label, set()).add(track_id)

    frame_class_counts = {}
    for d in detections_this_frame:
        frame_class_counts[d[4]] = frame_class_counts.get(d[4], 0) + 1

    num_detected = len(detections_this_frame)

    if num_detected > 0:
        no_animal_counter = 0
        if not animal_present:
            animal_present = True
            play_sound_async("alert_animal.wav")
            alert_timestamps_sec.append((frame_count / fps, "alert_animal.wav"))
            print(f"  🔔 [{frame_count}] ANIMAL DETECTED — 1 beep | {frame_class_counts}")
    else:
        no_animal_counter += 1
        if animal_present and no_animal_counter >= CLEAR_GRACE_FRAMES:
            animal_present    = False
            no_animal_counter = 0
            play_sound_async("alert_clear.wav")
            play_sound_async("alert_clear.wav", delay=0.05)
            alert_timestamps_sec.append((frame_count / fps,       "alert_clear.wav"))
            alert_timestamps_sec.append((frame_count / fps + 0.5, "alert_clear.wav"))
            print(f"  🔔🔔 [{frame_count}] ROAD CLEAR — 2 beeps")

    if REGION_LINE_Y:
        cv2.line(frame, (0, REGION_LINE_Y), (w, REGION_LINE_Y), (0,255,255), 2)
        cv2.putText(frame, "Detection Zone", (10, REGION_LINE_Y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)

    for (x1, y1, x2, y2, label, conf, track_id, cls_id) in detections_this_frame:
        color      = get_color(cls_id)
        label_text = f"{label} #{track_id}  {conf:.2f}"
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(frame, (x1, y1-th-8), (x1+tw+4, y1), color, -1)
        cv2.putText(frame, label_text, (x1+2, y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, TEXT_COLOR, 1)

    if num_detected > 0:
        cv2.rectangle(frame, (0,0), (w, 60), (0, 0, 200), -1)
        parts   = [f"{lbl}x{cnt}" for lbl, cnt in frame_class_counts.items()]
        summary = "  ".join(parts[:4])
        msg     = f"⚠ ANIMAL ON ROAD: {summary}"
        cv2.putText(frame, msg, (10, 42),
                    cv2.FONT_HERSHEY_DUPLEX, 0.65, (255,255,255), 2)
    else:
        if not animal_present:
            cv2.rectangle(frame, (0,0), (w, 60), (0, 150, 0), -1)
            cv2.putText(frame, "✓ ROAD CLEAR", (10, 42),
                        cv2.FONT_HERSHEY_DUPLEX, 0.65, (255,255,255), 2)

    panel_lines = [f"In frame: {num_detected}"]
    for lbl, id_set in per_class_count.items():
        panel_lines.append(f"  {lbl}: {len(id_set)} unique total")

    panel_h = 22 * (len(panel_lines) + 1)
    cv2.rectangle(frame, (0, h - panel_h), (430, h), (0,0,0), -1)
    for i, line in enumerate(panel_lines):
        color = (0,255,0) if i == 0 else (0,220,255)
        cv2.putText(frame, line, (8, h - panel_h + 22 + i*22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.52, color, 1)

    cv2.putText(frame, f"Frame {frame_count}/{total_frames}",
                (w-200, h-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)

    video_writer.write(frame)

    if frame_count % 50 == 0:
        print(f"  {frame_count}/{total_frames} | {frame_class_counts}")

cap.release()
video_writer.release()

print(f"\n✅ Done!  Frames processed: {frame_count}")
print(f"   Unique animals by class:")
for lbl, id_set in per_class_count.items():
    print(f"     {lbl:25s} → {len(id_set)} individual(s)")
print(f"   Output: {VIDEO_OUTPUT}")

In [ ]:
import numpy as np
import wave
import subprocess

SAMPLE_RATE   = 44100
total_seconds = total_frames / fps

def load_wav(filepath):
    with wave.open(filepath, 'r') as wf:
        frames = wf.readframes(wf.getnframes())
        return np.frombuffer(frames, dtype=np.int16).astype(np.float32)

beep_animal = load_wav("alert_animal.wav")
beep_clear  = load_wav("alert_clear.wav")

beep_map = {
    "alert_animal.wav" : beep_animal,
    "alert_clear.wav"  : beep_clear,
}

n_total   = int(total_seconds * SAMPLE_RATE) + SAMPLE_RATE
audio_out = np.zeros(n_total, dtype=np.float32)

for (t_sec, sound_file) in alert_timestamps_sec:
    beep   = beep_map[sound_file]
    start  = int(t_sec * SAMPLE_RATE)
    end    = min(start + len(beep), n_total)
    length = end - start
    audio_out[start:end] += beep[:length]

peak = np.max(np.abs(audio_out))
if peak > 0:
    audio_out = audio_out / peak * 32000

audio_int16 = audio_out.astype(np.int16)

with wave.open("combined_alerts.wav", "w") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(SAMPLE_RATE)
    wf.writeframes(audio_int16.tobytes())

print(f"Audio track built ✅  ({len(alert_timestamps_sec)} alert(s) stamped)")

FINAL_OUTPUT = "animal_detection_with_audio.mp4"

result = subprocess.run([
    "ffmpeg", "-y",
    "-i", VIDEO_OUTPUT,
    "-i", "combined_alerts.wav",
    "-c:v", "copy",
    "-c:a", "aac",
    "-b:a", "128k",
    "-shortest",
    FINAL_OUTPUT
], capture_output=True, text=True)

if result.returncode == 0:
    print(f"Mux complete ✅  →  {FINAL_OUTPUT}")
else:
    print("FFmpeg error ❌")
    print(result.stderr[-2000:])

from google.colab import files
files.download(FINAL_OUTPUT)
print("Download started ⬇️")
